# Glue → Salient

Cross-type attention: how much **Glue** tokens attend to **Salient** tokens across all 24 heads.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch as t
from IPython.display import Markdown, display
from shared import (
    load_model, run_and_cache, get_attention_pattern,
    show_head_pattern, show_attention_tables,
    get_type_positions, _values_entropy_normalized,
    show_cross_tokens, show_top_cross_pairs,
    CROSS_TYPE_NAMES,
)

/home/burny/projects/ai/mechinterp/attention-head-zoo-2-layer-attention-only-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = load_model()
str_tokens, logits, cache = run_and_cache(model)

## Matched Tokens

In [3]:
show_cross_tokens(str_tokens, "glue", "salient")

**From (Glue) tokens (25):** `We` (1), ` that` (3), ` is` (11), ` more` (12), ` than` (14), ` to` (16), ` be` (17), ` this` (19), ` If` (22), ` were` (27), ` to` (30), ` this` (31), ` we` (34), ` they` (36), ` by` (38), ` that` (42), ` are` (43), ` or` (45), ` and` (49), ` that` (50), ` are` (54), ` for` (56), ` how` (57), ` to` (58), ` this` (60)

**To (Salient) tokens (19):** ` think` (2), ` powerful` (4), ` significantly` (6), ` super` (7), `human` (8), ` machine` (9), ` intelligence` (10), ` century` (20), ` machine` (24), ` learning` (25), ` scaled` (28), ` level` (32), ` think` (35), ` systems` (41), ` deceptive` (44), ` manip` (46), `ulative` (47), ` plans` (53), ` avoid` (59)

## Glue → Salient — All 24 Heads

In [4]:
type_positions = get_type_positions(str_tokens)
from_pos = type_positions["glue"]
to_pos = type_positions["salient"]
if not from_pos or not to_pos:
    print("No positions found for one or both types.")
else:
    results = []
    for layer in range(2):
        for head in range(12):
            a = get_attention_pattern(cache, layer, head)
            pct = a[from_pos][:, to_pos].sum().item() / a.sum().item() * 100
            values = a[from_pos][:, to_pos].sum(dim=-1)
            ent = _values_entropy_normalized(values) * 100
            results.append(((layer, head), pct, ent))
    results.sort(key=lambda x: x[1], reverse=True)
    lines = ["| Head | Glue → Salient % | Entropy % |", "|------|-------|-------|"]  
    for (l, h), pct, ent in results:
        lines.append(f"| L{l}H{h} | {pct:.2f}% | {ent:.2f}% |")
    display(Markdown("\n".join(lines)))

| Head | Glue → Salient % | Entropy % |
|------|-------|-------|
| L1H5 | 22.19% | 97.61% |
| L0H2 | 20.83% | 97.89% |
| L1H9 | 20.75% | 97.38% |
| L1H7 | 16.98% | 96.46% |
| L0H6 | 13.36% | 96.98% |
| L0H8 | 13.26% | 95.49% |
| L0H4 | 12.95% | 92.71% |
| L1H2 | 12.78% | 90.28% |
| L0H7 | 12.05% | 64.25% |
| L1H0 | 11.11% | 93.19% |
| L1H1 | 10.02% | 94.85% |
| L0H11 | 9.58% | 88.65% |
| L1H6 | 8.71% | 77.34% |
| L0H0 | 8.61% | 86.82% |
| L0H1 | 8.55% | 96.45% |
| L0H9 | 6.56% | 78.90% |
| L0H10 | 6.19% | 91.30% |
| L1H3 | 6.02% | 92.16% |
| L1H10 | 5.56% | 83.19% |
| L1H8 | 4.68% | 68.25% |
| L0H5 | 2.80% | 79.45% |
| L0H3 | 2.39% | 96.32% |
| L1H11 | 2.28% | 79.77% |
| L1H4 | 1.87% | 82.33% |

## Top Heads

In [5]:
if from_pos and to_pos:
    for (l, h), pct, ent in results[:3]:
        display(Markdown(f"---\n### L{l}H{h} — {pct:.2f}% (ent {ent:.2f}%)"))
        show_head_pattern(str_tokens, cache, layer=l, head=h)
        display(Markdown("#### Top Glue → Salient token pairs"))
        show_top_cross_pairs(str_tokens, cache, l, h, from_pos, to_pos, top_k=10)
        attention = get_attention_pattern(cache, layer=l, head=h)
        show_attention_tables(str_tokens, attention, top_k=15)

---
### L1H5 — 22.19% (ent 97.61%)

#### Top Glue → Salient token pairs

| From Token | To Token | Attention |
|-----------|----------|-----------|
| ` more` (12) | ` think` (2) | 0.3178 |
| ` that` (3) | ` think` (2) | 0.2931 |
| ` to` (16) | ` think` (2) | 0.2874 |
| ` is` (11) | ` think` (2) | 0.2466 |
| ` more` (12) | ` powerful` (4) | 0.2145 |
| ` this` (31) | ` century` (20) | 0.2063 |
| ` than` (14) | `human` (8) | 0.1967 |
| ` is` (11) | `human` (8) | 0.1929 |
| ` this` (19) | `human` (8) | 0.1848 |
| ` were` (27) | ` think` (2) | 0.1677 |

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | `.` | `<\|endoftext\|>` | 61 | 0 | 0.9996 |
| 3 | `.` | `<\|endoftext\|>` | 21 | 0 | 0.9951 |
| 4 | `We` | `We` | 1 | 1 | 0.8676 |
| 5 | ` that` | ` that` | 3 | 3 | 0.6319 |
| 6 | ` think` | `We` | 2 | 1 | 0.5767 |
| 7 | ` significantly` | ` significantly` | 6 | 6 | 0.4482 |
| 8 | ` for` | ` known` | 56 | 55 | 0.3345 |
| 9 | `,` | `We` | 5 | 1 | 0.3265 |
| 10 | `human` | `human` | 8 | 8 | 0.3241 |
| 11 | ` more` | ` think` | 12 | 2 | 0.3178 |
| 12 | ` powerful` | ` that` | 4 | 3 | 0.3109 |
| 13 | ` that` | ` think` | 3 | 2 | 0.2931 |
| 14 | ` to` | ` think` | 16 | 2 | 0.2874 |
| 15 | ` powerful` | ` powerful` | 4 | 4 | 0.2741 |

...

| From Token | To Token | Attention |
|-----------|----------|-----------|
| ` more` (12) | ` powerful` (4) | 0.1954 |
| ` this` (19) | ` machine` (9) | 0.1890 |
| ` If` (22) | `human` (8) | 0.1598 |
| ` If` (22) | ` intelligence` (10) | 0.1402 |
| ` this` (19) | ` significantly` (6) | 0.1371 |
| ` If` (22) | ` machine` (9) | 0.1283 |
| ` this` (19) | ` intelligence` (10) | 0.1267 |
| ` more` (12) | ` significantly` (6) | 0.1240 |
| ` how` (57) | ` intelligence` (10) | 0.1225 |
| ` than` (14) | `human` (8) | 0.1207 |

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | ` manip` | `<\|endoftext\|>` | 46 | 0 | 0.6480 |
| 3 | ` that` | `<\|endoftext\|>` | 3 | 0 | 0.6478 |
| 4 | ` think` | `<\|endoftext\|>` | 2 | 0 | 0.6242 |
| 5 | `We` | `<\|endoftext\|>` | 1 | 0 | 0.6070 |
| 6 | ` intelligence` | `<\|endoftext\|>` | 10 | 0 | 0.5625 |
| 7 | `human` | `<\|endoftext\|>` | 8 | 0 | 0.5567 |
| 8 | ` super` | `<\|endoftext\|>` | 7 | 0 | 0.5019 |
| 9 | ` machine` | `<\|endoftext\|>` | 9 | 0 | 0.4962 |
| 10 | ` to` | `<\|endoftext\|>` | 30 | 0 | 0.3982 |
| 11 | ` significantly` | `<\|endoftext\|>` | 6 | 0 | 0.3960 |
| 12 | `We` | `We` | 1 | 1 | 0.3930 |
| 13 | ` powerful` | `<\|endoftext\|>` | 4 | 0 | 0.3223 |
| 14 | `,` | ` think` | 5 | 2 | 0.2979 |
| 15 | ` solid` | `<\|endoftext\|>` | 52 | 0 | 0.2603 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | ` manip` | ` would` | 46 | 37 | 0.0001 |
| 2 | ` manip` | ` think` | 46 | 35 | 0.0001 |
| 3 | ` produce` | ` think` | 40 | 35 | 0.0002 |
| 4 | ` plans` | ` think` | 53 | 35 | 0.0002 |
| 5 | ` manip` | ` likely` | 46 | 13 | 0.0002 |
| 6 | ` manip` | ` up` | 46 | 29 | 0.0002 |
| 7 | ` up` | ` up` | 29 | 29 | 0.0002 |
| 8 | ` plans` | ` up` | 53 | 29 | 0.0002 |
| 9 | ` to` | ` think` | 58 | 35 | 0.0002 |
| 10 | ` produce` | ` up` | 40 | 29 | 0.0002 |
| 11 | ` plans` | ` likely` | 53 | 13 | 0.0003 |
| 12 | ` to` | ` would` | 58 | 37 | 0.0003 |
| 13 | ` avoid` | ` would` | 59 | 37 | 0.0003 |
| 14 | ` manip` | ` significantly` | 46 | 6 | 0.0004 |
| 15 | ` produce` | ` likely` | 40 | 13 | 0.0004 |